In [1]:
%pip install streamlit pymupdf


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit as st
import pymupdf

print("DocuTrace AI setup successful!")
print("Streamlit:", st.__version__)
print("PyMuPDF:", pymupdf.VersionBind)

DocuTrace AI setup successful!
Streamlit: 1.66.1
PyMuPDF: 1.28.2


In [3]:
import pymupdf
from pathlib import Path

print("PDF extraction module is ready!")

PDF extraction module is ready!


In [4]:
pdf_path = Path("Notice_A.pdf")

doc = pymupdf.open(pdf_path)

for page_number, page in enumerate(doc, start=1):
    text = page.get_text("text")
    print(f"Page {page_number}:")
    print(text)

doc.close()

Page 1:
The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript.



In [5]:
def extract_pdf_text(pdf_path):
    """Extract text from each page of a PDF."""

    extracted_pages = []

    with pymupdf.open(pdf_path) as doc:
        if doc.needs_pass:
            raise ValueError("PDF is password-protected.")

        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()

            if text:
                extracted_pages.append({
                    "document": Path(pdf_path).name,
                    "page": page_number,
                    "text": text
                })

    return extracted_pages


pages = extract_pdf_text("Notice_A.pdf")

for item in pages:
    print(item)

{'document': 'Notice_A.pdf', 'page': 1, 'text': 'The application deadline for the internship is 15 November 2026. Applicants must submit their \nresume and academic transcript.'}


In [6]:
pdf_files = [
    "Notice_A.pdf",
    "Notice_B.pdf"
]

all_pages = []

for pdf_file in pdf_files:
    all_pages.extend(extract_pdf_text(pdf_file))

print("Total extracted pages:", len(all_pages))

for item in all_pages:
    print("\nDocument:", item["document"])
    print("Page:", item["page"])
    print("Text:", item["text"])


Total extracted pages: 2

Document: Notice_A.pdf
Page: 1
Text: The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript.

Document: Notice_B.pdf
Page: 1
Text: The application deadline for the internship is 20 November 2026. Applicants must submit their 
resume and academic transcript.


In [7]:
print("Total pages:", len(all_pages))
print(all_pages[0])

Total pages: 2
{'document': 'Notice_A.pdf', 'page': 1, 'text': 'The application deadline for the internship is 15 November 2026. Applicants must submit their \nresume and academic transcript.'}


In [8]:
import re

In [9]:
def tokenize(text):
    """Convert text into lowercase words."""

    return re.findall(r"\b[a-z0-9]+\b", text.lower())


print(tokenize("What is the Application Deadline?"))

['what', 'is', 'the', 'application', 'deadline']


In [10]:
def retrieve_evidence(question, pages, top_k=5):
    """
    Retrieve passages that share meaningful words
    with the user's question.
    """

    stop_words = {
        "what", "when", "where", "who", "why", "how",
        "is", "are", "was", "were", "the", "a", "an",
        "of", "in", "on", "for", "to", "and", "or",
        "does", "do", "did", "please", "tell", "me"
    }

    question_words = set(tokenize(question)) - stop_words

    if not question_words:
        return []

    results = []

    for page_data in pages:
        passage_words = set(tokenize(page_data["text"]))
        matched_words = question_words & passage_words

        if matched_words:
            score = len(matched_words) / len(question_words)

            results.append({
                "document": page_data["document"],
                "page": page_data["page"],
                "text": page_data["text"],
                "score": round(score, 3),
                "matched_words": sorted(matched_words)
            })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]

In [11]:
question = "What is the application deadline?"

evidence = retrieve_evidence(
    question,
    all_pages,
    top_k=5
)

for item in evidence:
    print("-" * 50)
    print("Source:", item["document"])
    print("Page:", item["page"])
    print("Relevance score:", item["score"])
    print("Matched words:", item["matched_words"])
    print("Evidence:", item["text"])

--------------------------------------------------
Source: Notice_A.pdf
Page: 1
Relevance score: 1.0
Matched words: ['application', 'deadline']
Evidence: The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript.
--------------------------------------------------
Source: Notice_B.pdf
Page: 1
Relevance score: 1.0
Matched words: ['application', 'deadline']
Evidence: The application deadline for the internship is 20 November 2026. Applicants must submit their 
resume and academic transcript.


In [12]:
question = "What is the company's office address?"

evidence = retrieve_evidence(
    question,
    all_pages,
    top_k=5
)

if not evidence:
    print("No matching evidence found.")

else:
    for item in evidence:
        print(item["document"], item["page"])
        print(item["text"])

No matching evidence found.


In [13]:
def display_evidence(evidence):
    if not evidence:
        print("No relevant evidence found.")
        return

    for index, item in enumerate(evidence, start=1):
        print(f"\nEvidence {index}")
        print(f"Source: {item['document']}")
        print(f"Page: {item['page']}")
        print(f"Score: {item['score']}")
        print(f"Passage: {item['text']}")


display_evidence(
    retrieve_evidence(
        "What is the application deadline?",
        all_pages
    )
)


Evidence 1
Source: Notice_A.pdf
Page: 1
Score: 1.0
Passage: The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript.

Evidence 2
Source: Notice_B.pdf
Page: 1
Score: 1.0
Passage: The application deadline for the internship is 20 November 2026. Applicants must submit their 
resume and academic transcript.


In [14]:
def answer_question_free(question, pages, top_k=5):
    """
    Free, rule-based answer engine for DocuTrace AI.
    Uses the existing Phase 3 retrieval function.
    Requires no API key or language model download.
    """

    evidence = retrieve_evidence(
        question,
        pages,
        top_k=top_k
    )

    if not evidence:
        return {
            "status": "insufficient_evidence",
            "answer": (
                "I could not find relevant evidence "
                "in the uploaded documents."
            ),
            "citations": []
        }

    citations = []

    for item in evidence:
        citations.append({
            "document": item["document"],
            "page": item["page"],
            "passage": item["text"],
            "score": item["score"]
        })

    # Avoid presenting a single answer if multiple
    # relevant passages contain potentially different facts.
    answer_lines = [
        "Relevant information found in the documents:"
    ]

    for item in citations:
        answer_lines.append(
            f"- {item['passage']} "
            f"[Source: {item['document']}, "
            f"Page {item['page']}]"
        )

    return {
        "status": "evidence_found",
        "answer": "\n".join(answer_lines),
        "citations": citations
    }

In [15]:
result = answer_question_free(
    "What is the application deadline?",
    all_pages
)

print("STATUS:", result["status"])
print("\nANSWER:\n")
print(result["answer"])

print("\nCITATIONS:")
for citation in result["citations"]:
    print(
        citation["document"],
        "| Page:", citation["page"],
        "| Score:", citation["score"]
    )

STATUS: evidence_found

ANSWER:

Relevant information found in the documents:
- The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript. [Source: Notice_A.pdf, Page 1]
- The application deadline for the internship is 20 November 2026. Applicants must submit their 
resume and academic transcript. [Source: Notice_B.pdf, Page 1]

CITATIONS:
Notice_A.pdf | Page: 1 | Score: 1.0
Notice_B.pdf | Page: 1 | Score: 1.0


In [16]:
import re
from collections import defaultdict

MONTHS = (
    "January|February|March|April|May|June|July|"
    "August|September|October|November|December|"
    "Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec"
)

DATE_PATTERN = re.compile(
    rf"\b(?:\d{{1,2}}\s+(?:{MONTHS})\.?"
    rf"|(?:{MONTHS})\s+\d{{1,2}})"
    rf"(?:,?\s+\d{{4}})?\b",
    re.IGNORECASE
)

TOPIC_KEYWORDS = {
    "application deadline": [
        "deadline", "last date", "apply by", "due date"
    ],
    "event date": [
        "event date", "event will be held", "scheduled for"
    ],
    "exam date": [
        "exam date", "examination", "test will be held"
    ],
    "interview date": [
        "interview date", "interview will be held"
    ]
}


def detect_date_conflicts(pages):
    """
    Detect different date strings associated with the
    same supported topic across document passages.
    """

    findings = defaultdict(list)

    for item in pages:
        text = item["text"]

        for topic, keywords in TOPIC_KEYWORDS.items():
            for sentence in re.split(r"(?<=[.!?])\s+", text):
                sentence_lower = sentence.lower()

                if any(word in sentence_lower for word in keywords):
                    dates = DATE_PATTERN.findall(sentence)

                    for date in dates:
                        findings[topic].append({
                            "date": date.strip(),
                            "document": item["document"],
                            "page": item["page"],
                            "passage": sentence.strip()
                        })

    conflicts = []

    for topic, statements in findings.items():
        distinct_dates = {
            re.sub(r"\s+", " ", item["date"].lower())
            for item in statements
        }

        distinct_documents = {
            item["document"] for item in statements
        }

        if len(distinct_dates) > 1 and len(distinct_documents) > 1:
            conflicts.append({
                "topic": topic,
                "statements": statements
            })

    return conflicts

In [17]:
conflicts = detect_date_conflicts(all_pages)

if not conflicts:
    print("No date conflicts detected by the current rules.")

else:
    for conflict in conflicts:
        print("=" * 60)
        print("POTENTIAL CONFLICT:", conflict["topic"])
        print("-" * 60)

        for statement in conflict["statements"]:
            print(
                f"Source: {statement['document']} "
                f"(Page {statement['page']})"
            )
            print("Evidence:", statement["passage"])
            print()

POTENTIAL CONFLICT: application deadline
------------------------------------------------------------
Source: Notice_A.pdf (Page 1)
Evidence: The application deadline for the internship is 15 November 2026.

Source: Notice_B.pdf (Page 1)
Evidence: The application deadline for the internship is 20 November 2026.



In [18]:
def analyze_documents_free(question, pages):
    answer = answer_question_free(question, pages)
    conflicts = detect_date_conflicts(pages)

    return {
        "question": question,
        "answer": answer,
        "conflicts": conflicts
    }


report = analyze_documents_free(
    "What is the application deadline?",
    all_pages
)

print("QUESTION:", report["question"])
print("\nANSWER:")
print(report["answer"]["answer"])

print("\nPOTENTIAL CONFLICTS:")
if report["conflicts"]:
    for conflict in report["conflicts"]:
        print(
            conflict["topic"],
            "->",
            len(conflict["statements"]),
            "supporting statements"
        )
else:
    print("No conflicts found by the current rules.")

QUESTION: What is the application deadline?

ANSWER:
Relevant information found in the documents:
- The application deadline for the internship is 15 November 2026. Applicants must submit their 
resume and academic transcript. [Source: Notice_A.pdf, Page 1]
- The application deadline for the internship is 20 November 2026. Applicants must submit their 
resume and academic transcript. [Source: Notice_B.pdf, Page 1]

POTENTIAL CONFLICTS:
application deadline -> 2 supporting statements


In [19]:
test_pages = [
    {
        "document": "Notice_A.pdf",
        "page": 1,
        "text": (
            "The application deadline is "
            "15 November 2026. "
            "The registration fee is Rs. 500."
        )
    },
    {
        "document": "Notice_B.pdf",
        "page": 1,
        "text": (
            "The application deadline is "
            "20 November 2026. "
            "The registration fee is Rs. 700."
        )
    },
    {
        "document": "Notice_C.pdf",
        "page": 2,
        "text": (
            "Applicants must submit their resume."
        )
    }
]

print("Test documents:", len(test_pages))

Test documents: 3


In [20]:
import re
from collections import defaultdict


def extract_attributes(text):
    """Extract simple dates and monetary amounts."""

    attributes = {}

    # Dates such as 15 November 2026 or November 15, 2026
    date_pattern = (
        r"\b\d{1,2}\s+"
        r"(?:January|February|March|April|May|June|July|"
        r"August|September|October|November|December)"
        r"\s+\d{4}\b"
        r"|\b(?:January|February|March|April|May|June|July|"
        r"August|September|October|November|December)"
        r"\s+\d{1,2},?\s+\d{4}\b"
    )

    dates = re.findall(
        date_pattern, text, flags=re.IGNORECASE
    )

    if dates:
        attributes["date"] = dates

    # Monetary amounts such as Rs. 500, INR 500, or ₹500
    money_pattern = (
        r"(?:Rs\.?|INR|₹)\s*[\d,]+(?:\.\d{1,2})?"
    )

    amounts = re.findall(
        money_pattern, text, flags=re.IGNORECASE
    )

    if amounts:
        attributes["amount"] = amounts

    return attributes


print(extract_attributes(test_pages[0]["text"]))

{'date': ['15 November 2026'], 'amount': ['Rs. 500']}


In [21]:
TOPIC_RULES = {
    "application_deadline": [
        "application deadline",
        "last date to apply",
        "deadline for application",
        "apply by"
    ],
    "registration_fee": [
        "registration fee",
        "application fee",
        "registration charge"
    ],
    "exam_date": [
        "exam date",
        "examination date",
        "exam will be held"
    ],
    "interview_date": [
        "interview date",
        "interview will be held",
        "interview scheduled"
    ]
}


def identify_topics(text):
    text_lower = text.lower()

    return [
        topic
        for topic, phrases in TOPIC_RULES.items()
        if any(phrase in text_lower for phrase in phrases)
    ]


print(identify_topics(
    "The application deadline is 15 November 2026."
))

print(identify_topics(
    "The registration fee is Rs. 500."
))

['application_deadline']
['registration_fee']


In [22]:
def detect_conflicts_v2(pages):
    """
    Detect potential conflicts in dates and monetary amounts.

    Each result retains the source filename, page number,
    and original text passage.
    """

    grouped_facts = defaultdict(list)

    for item in pages:
        text = item["text"]

        # Analyze individual sentences to keep evidence focused.
        sentences = re.split(r"(?<=[.!?])\s+", text)

        for sentence in sentences:
            topics = identify_topics(sentence)
            attributes = extract_attributes(sentence)

            for topic in topics:
                for attribute, values in attributes.items():
                    for value in values:
                        grouped_facts[(topic, attribute)].append({
                            "topic": topic,
                            "attribute": attribute,
                            "value": value,
                            "document": item["document"],
                            "page": item["page"],
                            "passage": sentence.strip()
                        })

    conflicts = []

    for (topic, attribute), facts in grouped_facts.items():

        # Compare normalized values, not raw capitalization.
        distinct_values = {
            re.sub(r"\s+", " ", fact["value"])
            .strip()
            .lower()
            for fact in facts
        }

        distinct_documents = {
            fact["document"] for fact in facts
        }

        if (
                len(distinct_values) > 1
                and len(distinct_documents) > 1
        ):
            conflicts.append({
                "topic": topic,
                "attribute": attribute,
                "statements": facts,
                "values": sorted(distinct_values),
                "status": "potential_conflict"
            })

    return conflicts

In [23]:
conflicts = detect_conflicts_v2(test_pages)

if not conflicts:
    print("No potential conflicts detected.")

else:
    for conflict in conflicts:
        print("=" * 60)
        print("TOPIC:", conflict["topic"])
        print("ATTRIBUTE:", conflict["attribute"])
        print("STATUS:", conflict["status"])

        for statement in conflict["statements"]:
            print(
                f"\nSource: {statement['document']}"
                f" | Page: {statement['page']}"
            )
            print("Value:", statement["value"])
            print("Evidence:", statement["passage"])

TOPIC: application_deadline
ATTRIBUTE: date
STATUS: potential_conflict

Source: Notice_A.pdf | Page: 1
Value: 15 November 2026
Evidence: The application deadline is 15 November 2026.

Source: Notice_B.pdf | Page: 1
Value: 20 November 2026
Evidence: The application deadline is 20 November 2026.


In [24]:
def print_conflict_report(conflicts):
    print("=" * 64)
    print("             DOCTRACE AI")
    print("       CROSS-DOCUMENT CONFLICT REPORT")
    print("=" * 64)

    if not conflicts:
        print("No conflicts detected by the current rules.")
        return

    print("Potential conflicts detected:", len(conflicts))

    for number, conflict in enumerate(conflicts, start=1):
        print("\n" + "-" * 64)
        print(f"CONFLICT {number}: {conflict['topic']}")
        print(f"Attribute: {conflict['attribute']}")
        print("Status: POTENTIAL CONFLICT")

        for statement in conflict["statements"]:
            print(
                f"\nSource: {statement['document']}"
                f" | Page {statement['page']}"
            )
            print(f"Value: {statement['value']}")
            print(f"Evidence: {statement['passage']}")

        print(
            "\nRecommended action: Verify the relevant "
            "documents, their publication dates, and "
            "the issuing authority."
        )

    print("\n" + "=" * 64)


print_conflict_report(conflicts)

             DOCTRACE AI
       CROSS-DOCUMENT CONFLICT REPORT
Potential conflicts detected: 1

----------------------------------------------------------------
CONFLICT 1: application_deadline
Attribute: date
Status: POTENTIAL CONFLICT

Source: Notice_A.pdf | Page 1
Value: 15 November 2026
Evidence: The application deadline is 15 November 2026.

Source: Notice_B.pdf | Page 1
Value: 20 November 2026
Evidence: The application deadline is 20 November 2026.

Recommended action: Verify the relevant documents, their publication dates, and the issuing authority.



In [25]:
real_conflicts = detect_conflicts_v2(all_pages)

print_conflict_report(real_conflicts)

             DOCTRACE AI
       CROSS-DOCUMENT CONFLICT REPORT
Potential conflicts detected: 1

----------------------------------------------------------------
CONFLICT 1: application_deadline
Attribute: date
Status: POTENTIAL CONFLICT

Source: Notice_A.pdf | Page 1
Value: 15 November 2026
Evidence: The application deadline for the internship is 15 November 2026.

Source: Notice_B.pdf | Page 1
Value: 20 November 2026
Evidence: The application deadline for the internship is 20 November 2026.

Recommended action: Verify the relevant documents, their publication dates, and the issuing authority.



In [26]:

%%writefile app.py
import re
from collections import defaultdict

import streamlit as st
import pymupdf

# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="DocuTrace AI",
    page_icon="📄",
    layout="wide"
)

st.title("📄 DocuTrace AI")
st.caption(
    "Evidence-Based Document Question Answering "
    "& Cross-Document Conflict Detection"
)

st.info(
    "Free prototype: Python-based retrieval and conflict "
    "rules. No paid API or downloaded language model."
)


# --------------------------------------------------
# 2. PDF TEXT EXTRACTION
# --------------------------------------------------

def extract_uploaded_pdf(uploaded_file):
    """Extract text while preserving source and page."""

    pages = []
    pdf_bytes = uploaded_file.getvalue()

    with pymupdf.open(
            stream=pdf_bytes,
            filetype="pdf"
    ) as doc:

        if doc.needs_pass:
            raise ValueError(
                "This PDF is password-protected."
            )

        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()

            if text:
                pages.append({
                    "document": uploaded_file.name,
                    "page": page_number,
                    "text": text
                })

    return pages


# --------------------------------------------------
# 3. EVIDENCE RETRIEVAL
# --------------------------------------------------

STOP_WORDS = {
    "what", "when", "where", "who", "why", "how",
    "is", "are", "was", "were", "the", "a", "an",
    "of", "in", "on", "for", "to", "and", "or",
    "does", "do", "did", "please", "tell", "me",
    "which", "from", "this", "that", "it"
}


def tokenize(text):
    return re.findall(r"\b[a-z0-9]+\b", text.lower())


def retrieve_evidence(question, pages, top_k=5):
    question_words = (
            set(tokenize(question)) - STOP_WORDS
    )

    if not question_words:
        return []

    results = []

    for item in pages:
        passage_words = set(tokenize(item["text"]))
        matched = question_words & passage_words

        if matched:
            score = len(matched) / len(question_words)

            results.append({
                **item,
                "score": round(score, 3),
                "matched_words": sorted(matched)
            })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]


# --------------------------------------------------
# 4. CONFLICT DETECTION RULES
# --------------------------------------------------

TOPIC_RULES = {
    "Application deadline": [
        "application deadline",
        "deadline for application",
        "last date to apply",
        "apply by"
    ],
    "Registration fee": [
        "registration fee",
        "application fee",
        "registration charge"
    ],
    "Exam date": [
        "exam date",
        "examination date",
        "exam will be held"
    ],
    "Interview date": [
        "interview date",
        "interview will be held",
        "interview scheduled"
    ],
    "Event date": [
        "event date",
        "event will be held",
        "event scheduled"
    ]
}

MONTHS = (
    "January|February|March|April|May|June|July|"
    "August|September|October|November|December|"
    "Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec"
)

DATE_PATTERN = re.compile(
    rf"\b(?:\d{{1,2}}\s+(?:{MONTHS})\.?"
    rf"|(?:{MONTHS})\s+\d{{1,2}})"
    rf"(?:,?\s+\d{{4}})?\b",
    re.IGNORECASE
)

MONEY_PATTERN = re.compile(
    r"(?:Rs\.?|INR|₹)\s*[\d,]+(?:\.\d{1,2})?",
    re.IGNORECASE
)


def detect_conflicts(pages):
    grouped = defaultdict(list)

    for item in pages:
        sentences = re.split(
            r"(?<=[.!?])\s+",
            item["text"]
        )

        for sentence in sentences:
            sentence_lower = sentence.lower()

            for topic, phrases in TOPIC_RULES.items():
                if not any(
                        phrase in sentence_lower
                        for phrase in phrases
                ):
                    continue

                # Look for date or money values.
                found_dates = DATE_PATTERN.findall(sentence)
                found_money = MONEY_PATTERN.findall(sentence)

                candidates = (
                        [("date", value) for value in found_dates]
                        + [("amount", value) for value in found_money]
                )

                for attribute, value in candidates:
                    grouped[(topic, attribute)].append({
                        "topic": topic,
                        "attribute": attribute,
                        "value": value.strip(),
                        "document": item["document"],
                        "page": item["page"],
                        "passage": sentence.strip()
                    })

    conflicts = []

    for (topic, attribute), facts in grouped.items():
        normalized_values = {
            re.sub(r"\s+", " ", fact["value"])
            .strip()
            .casefold()
            for fact in facts
        }

        distinct_documents = {
            fact["document"] for fact in facts
        }

        if (
                len(normalized_values) > 1
                and len(distinct_documents) > 1
        ):
            conflicts.append({
                "topic": topic,
                "attribute": attribute,
                "statements": facts,
                "status": "Potential conflict"
            })

    return conflicts


# --------------------------------------------------
# 5. USER INTERFACE: UPLOAD
# --------------------------------------------------

st.sidebar.header("Document workspace")

uploaded_files = st.sidebar.file_uploader(
    "Upload PDF documents",
    type=["pdf"],
    accept_multiple_files=True
)

pages = []
errors = []

if uploaded_files:
    for uploaded_file in uploaded_files:
        try:
            pages.extend(
                extract_uploaded_pdf(uploaded_file)
            )
        except Exception as error:
            errors.append(
                f"{uploaded_file.name}: {error}"
            )

for error in errors:
    st.sidebar.error(error)

document_names = sorted({
    item["document"] for item in pages
})

total_pages = len({
    (item["document"], item["page"])
    for item in pages
})

conflicts = detect_conflicts(pages)

# --------------------------------------------------
# 6. DASHBOARD METRICS
# --------------------------------------------------

col1, col2, col3 = st.columns(3)

col1.metric(
    "PDF documents",
    len(document_names)
)

col2.metric(
    "Pages with extracted text",
    total_pages
)

col3.metric(
    "Potential conflicts",
    len(conflicts)
)

if not uploaded_files:
    st.warning(
        "Upload one or more text-based PDFs using "
        "the sidebar to begin."
    )
    st.stop()

if not pages:
    st.error(
        "No selectable PDF text was extracted. "
        "Scanned PDFs may require OCR."
    )
    st.stop()

# --------------------------------------------------
# 7. QUESTION ANSWERING
# --------------------------------------------------

st.header("1. Ask your documents")

question = st.text_input(
    "Enter your question",
    placeholder="What is the application deadline?"
)

if st.button("Find evidence", type="primary"):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        evidence = retrieve_evidence(question, pages)

        if not evidence:
            st.warning(
                "No matching evidence was found. "
                "This keyword-based search may miss "
                "relevant passages."
            )
        else:
            st.subheader("Retrieved evidence")

            if len(evidence) > 1:
                st.caption(
                    "Multiple passages are shown. "
                    "Review them for possible disagreements."
                )

            for index, item in enumerate(evidence, start=1):
                with st.expander(
                        f"Evidence {index}: "
                        f"{item['document']} — Page {item['page']}",
                        expanded=True
                ):
                    st.write(item["text"])
                    st.caption(
                        f"Word-overlap score: {item['score']}"
                    )

            st.info(
                "This prototype retrieves evidence rather "
                "than generating a free-form LLM answer."
            )

# --------------------------------------------------
# 8. CONFLICT REPORT
# --------------------------------------------------

st.divider()
st.header("2. Cross-document conflict scan")

if not conflicts:
    st.success(
        "No potential conflicts detected by the current rules."
    )
    st.caption(
        "This does not prove the documents are consistent. "
        "Only supported topics and formats are checked."
    )
else:
    st.warning(
        f"{len(conflicts)} potential conflict(s) detected. "
        "Neither statement is automatically treated as correct."
    )

    for index, conflict in enumerate(conflicts, start=1):
        with st.expander(
                f"{index}. {conflict['topic']} "
                f"({conflict['attribute']})",
                expanded=True
        ):
            for statement in conflict["statements"]:
                st.markdown(
                    f"**Source:** {statement['document']} "
                    f"— Page {statement['page']}"
                )
                st.write("Value:", statement["value"])
                st.write("Evidence:", statement["passage"])
                st.divider()

            st.caption(
                "Recommended action: verify document versions "
                "or consult the issuing authority."
            )

# --------------------------------------------------
# 9. PPT-FRIENDLY SUMMARY CHART
# --------------------------------------------------

st.divider()
st.header("3. Analysis summary for project reporting")

if conflicts:
    topic_counts = defaultdict(int)

    for conflict in conflicts:
        topic_counts[conflict["topic"]] += 1

    st.subheader("Potential conflicts by topic")

    st.bar_chart(
        dict(topic_counts),
        horizontal=True
    )
else:
    st.write(
        "No conflict chart is available because no potential "
        "conflicts were detected in this scan."
    )

st.caption(
    "Results are generated by deterministic Python rules. "
    "False positives and missed conflicts are possible."
)


Writing app.py


In [27]:
import os

print(os.getcwd())
print(os.path.exists("app.py"))

/home/dream/github/Dream/Python/LLM using Python [Summer Internship - II]/DocuTrace
True
